## Getting exceedance times for each event

In [1]:
import pandas as pd
import numpy as np
# import event-tauc summary tables
spring_events = pd.read_csv("spring_events/spring_event_tauc_summary.csv", parse_dates=["event_start", "event_peak", "event_end", "first_exceedance_time"])
summer_events = pd.read_csv("summer_events/summer_event_tauc_summary.csv", parse_dates=["start", "peak_time", "end"])
tau = pd.read_csv("../data/shear_stress/average_total_shear_stress_corrected.csv", parse_dates=["datetime"])

# rename columns so spring and summer match
spring_events = spring_events.rename(columns={
    "event_start": "start",
    "event_peak": "peak_time",
    "event_end": "end",
})

# ensure all relevant columns have the correct type
for events in [spring_events, summer_events]:
    for col in ["start", "peak_time", "end", "first_exceedance_time"]:
        events[col] = pd.to_datetime(events[col], errors="coerce")

    events["first_exceedance_tau"] = pd.to_numeric(events["first_exceedance_tau"], errors="coerce")

tau = tau.set_index("datetime").sort_index()
tau_series = (pd.to_numeric(tau["shear_stress"], errors="coerce").groupby(level=0).mean().sort_index().dropna())

In [4]:
spring_events

,event,start,peak_time,end,has_tauc_exceedance,first_exceedance_time,first_exceedance_tau,n_transport_events,n_exceedance_points,total_exceedance_hours,max_Dmax_mm
0,0,2023-04-17 13:00:00,2023-04-17 22:18:00,2023-04-18 11:00:00,False,NaT,NaN,0,0,0.00,NaN
1,1,2023-04-18 11:00:00,2023-04-18 20:18:00,2023-04-19 10:00:00,False,NaT,NaN,0,0,0.00,NaN
2,2,2023-04-19 10:00:00,2023-04-19 18:03:00,2023-04-20 11:00:00,True,2023-04-19 15:45:00,118.227334,1,8,2.00,60.71043
3,3,2023-04-21 12:00:00,2023-04-21 18:33:00,2023-04-22 09:30:00,True,2023-04-21 22:45:00,108.511153,5,5,1.25,45.71622
4,4,2023-04-22 14:00:00,2023-04-22 23:18:00,2023-04-23 10:30:00,True,2023-04-22 15:30:00,106.970609,6,7,1.75,46.06356
5,5,2023-04-23 11:30:00,2023-04-23 20:18:00,2023-04-24 11:00:00,True,2023-04-23 17:15:00,113.720403,7,7,1.75,47.35500
6,6,2023-04-24 11:00:00,2023-04-24 17:48:00,2023-04-25 11:00:00,False,NaT,NaN,0,0,0.00,NaN
7,7,2023-04-27 12:00:00,2023-04-27 20:48:00,2023-04-28 12:00:00,True,2023-04-27 15:30:00,112.663199,8,9,2.25,101.10500
8,8,2023-04-29 12:00:00,2023-04-29 20:18:00,2023-04-30 12:00:00,True,2023-04-30 08:15:00,114.890976,5,5,1.25,44.46600
9,9,2023-04-30 12:00:00,2023-04-30 19:48:00,2023-05-01 11:00:00,True,2023-04-30 13:30:00,116.555317,6,24,6.00,47.50300


Defining functions

In [ ]:
def _get_tau_window_with_boundaries(tau_series, start, end):
    """
    extract tau between start and end and interpolate tau exactly at the
    start and end boundaries if/when those times are not in the original record.
    """
    if pd.isna(start) or pd.isna(end) or start >= end:
        return pd.Series(dtype=float)

    # need one measurement before and after the event boundaries
    before = tau_series.loc[:start].tail(1)
    within = tau_series.loc[start:end]
    after = tau_series.loc[end:].head(1)

    if before.empty or after.empty:
        return pd.Series(dtype=float)

    window = pd.concat([before, within, after])
    window = window[~window.index.duplicated(keep="last")].sort_index()

    # Add exact start and end times
    new_index = window.index.union(pd.DatetimeIndex([start, end])).sort_values()
    window = window.reindex(new_index)
    # interpolate only between existing measurements
    window = window.interpolate(method="time", limit_area="inside")
    return window.loc[start:end].dropna()

def integrate_positive_excess(tau_series, tauc, start, end, max_gap="20min"):
    """
    Calculate the integral of max(tau - tauc, 0) through time and duration for which tau is above tauc;
    when the threshold is crossed between measurements, we use linear interpolation
    """
    window = _get_tau_window_with_boundaries(tau_series, start, end)

    if len(window) < 2 or pd.isna(tauc):
        return {
            "integral": np.nan,
            "duration_h": np.nan,
            "covered_h": 0.0,
            "n_skipped_intervals": 0
        }

    max_gap = pd.Timedelta(max_gap) if max_gap is not None else None
    integral = 0.0
    duration_h = 0.0
    covered_h = 0.0
    n_skipped = 0
    times = window.index
    excess = window.to_numpy(dtype=float) - float(tauc)

    for i in range(len(window) - 1):
        interval = times[i + 1] - times[i]
        dt_h = interval.total_seconds() / 3600

        if dt_h <= 0:
            continue

        # do not integrate across large data gaps
        if max_gap is not None and interval > max_gap:
            n_skipped += 1
            continue
        y0 = excess[i]
        y1 = excess[i + 1]

        if not np.isfinite(y0) or not np.isfinite(y1):
            continue
        covered_h += dt_h

        # both endpoints at or below tauc
        if y0 <= 0 and y1 <= 0:
            continue

        # both endpoints at or above tauc
        if y0 >= 0 and y1 >= 0:
            duration_h += dt_h
            integral += 0.5 * (y0 + y1) * dt_h
            continue

        # threshold crossed upward during the interval
        if y0 < 0 < y1:
            fraction_before_crossing = -y0 / (y1 - y0)
            positive_dt_h = dt_h * (1 - fraction_before_crossing)

            duration_h += positive_dt_h

            # triangle above the threshold
            integral += 0.5 * y1 * positive_dt_h
            continue

        # threshold crossed downward during the interval
        if y0 > 0 > y1:
            fraction_above = y0 / (y0 - y1)
            positive_dt_h = dt_h * fraction_above

            duration_h += positive_dt_h

            # triangle above the threshold
            integral += 0.5 * y0 * positive_dt_h

    return {
        "integral": integral,
        "duration_h": duration_h,
        "covered_h": covered_h,
        "n_skipped_intervals": n_skipped
    }

def calculate_event_exceedance_metrics(events, tau_series, max_gap="20min"):
    """
    calculate hydraulic exceedance metrics for every event.
    first_exceedance_tau is treated as the event-specific tauc50.
    """

    tau_series = tau_series.sort_index().dropna()
    output_rows = []
    for _, row in events.iterrows():
        result = {}
        start = row["start"]
        supplied_peak_time = row["peak_time"]
        end = row["end"]

        # event-specific observed critical shear stress
        tauc = row["first_exceedance_tau"]
        first_motion_time = row.get("first_exceedance_time", pd.NaT)
        result["tauc50"] = tauc

        # quality-control check
        result["first_motion_in_event"] = (pd.notna(first_motion_time) and pd.notna(start) and pd.notna(end) and start <= first_motion_time <= end)
        if (pd.isna(start) or pd.isna(end) or start >= end):
            result["calculation_status"] = "invalid event window"
            output_rows.append(result)
            continue
        if pd.isna(tauc):
            # no observed D50 transport means there is no measured tauc50
            result["calculation_status"] = "no measured tauc50"
            output_rows.append(result)
            continue
        event_tau = tau_series.loc[start:end].dropna()
        if event_tau.empty:
            result["calculation_status"] = "no tau data in event"
            output_rows.append(result)
            continue

        # peak from the complete shear-stress record
        tau_peak = event_tau.max()
        tau_peak_time_record = event_tau.idxmax()
        result["tau_peak"] = tau_peak
        result["tau_peak_time_record"] = tau_peak_time_record
        result["peak_mobility_ratio"] = (tau_peak / tauc if tauc != 0 else np.nan)
        result["max_excess_tau"] = max(tau_peak - tauc, 0)
        result["tau_reached_tauc"] = bool(tau_peak >= tauc)

        # use supplied event peak for limb separation when valid.
        # otherwise, use the maximum in the tau record.
        if (pd.notna(supplied_peak_time) and start <= supplied_peak_time <= end):
            split_peak_time = supplied_peak_time
            result["peak_time_source"] = "event table"
        else:
            split_peak_time = tau_peak_time_record
            result["peak_time_source"] = "tau record maximum"
        result["split_peak_time_used"] = split_peak_time

        # timing of observed D50 transport relative to the hydrograph peak
        if pd.notna(first_motion_time):
            result["first_motion_relative_to_peak_h"] = (first_motion_time - split_peak_time).total_seconds() / 3600
        else:
            result["first_motion_relative_to_peak_h"] = np.nan

        rising = integrate_positive_excess(tau_series=tau_series, tauc=tauc, start=start, end=split_peak_time, max_gap=max_gap)
        falling = integrate_positive_excess(tau_series=tau_series, tauc=tauc, start=split_peak_time, end=end, max_gap=max_gap)

        E_rise = rising["integral"]
        E_fall = falling["integral"]

        T_rise = rising["duration_h"]
        T_fall = falling["duration_h"]

        E_total = E_rise + E_fall
        T_total = T_rise + T_fall

        result["excess_integral_rise"] = E_rise
        result["excess_integral_fall"] = E_fall
        result["excess_integral_total"] = E_total

        result["tau_above_tauc_h_rise"] = T_rise
        result["tau_above_tauc_h_fall"] = T_fall
        result["tau_above_tauc_h_total"] = T_total

        result["mean_excess_tau"] = (E_total / T_total if T_total > 0 else np.nan) # average amount by which tau exceeded tauc while above threshold
        result["excess_fraction_rise"] = (E_rise / E_total if E_total > 0 else np.nan) # fraction of cumulative exceedance on each limb
        result["excess_fraction_fall"] = (E_fall / E_total if E_total > 0 else np.nan)

        # -1 = completely rising-limb dominated
        # +1 = completely falling-limb dominated
        result["excess_balance"] = ((E_fall - E_rise) / E_total if E_total > 0 else np.nan)

        # equivalent fractions based only on exceedance duration
        result["duration_fraction_rise"] = (T_rise / T_total if T_total > 0 else np.nan)
        result["duration_fraction_fall"] = (T_fall / T_total if T_total > 0 else np.nan)
        result["duration_balance"] = ((T_fall - T_rise) / T_total if T_total > 0 else np.nan)

        # tau-record coverage and gap checks
        covered_h = (rising["covered_h"] + falling["covered_h"])
        event_length_h = (end - start).total_seconds() / 3600
        result["tau_record_coverage_fraction"] = (covered_h / event_length_h if event_length_h > 0 else np.nan)
        result["n_skipped_tau_intervals"] = (rising["n_skipped_intervals"] + falling["n_skipped_intervals"])

        # first point in the full tau record that reaches the threshold
        tau_at_or_above = event_tau[event_tau >= tauc]
        if not tau_at_or_above.empty:
            first_record_threshold_time = tau_at_or_above.index[0]
            result["first_tau_ge_tauc_time_record"] = (first_record_threshold_time)
            if pd.notna(first_motion_time):
                result["record_threshold_minus_observed_motion_min"] = (first_record_threshold_time - first_motion_time).total_seconds() / 60
            else:
                result["record_threshold_minus_observed_motion_min"] = np.nan
        else:
            result["first_tau_ge_tauc_time_record"] = pd.NaT
            result["record_threshold_minus_observed_motion_min"] = np.nan
        result["calculation_status"] = "calculated"
        output_rows.append(result)

    metrics = pd.DataFrame(output_rows)
    return pd.concat([events.reset_index(drop=True), metrics.reset_index(drop=True)], axis=1)



Calculate exceedance metrics for each season:

In [5]:
spring_exceedance = calculate_event_exceedance_metrics(spring_events, tau_series, max_gap="20min")
summer_exceedance = calculate_event_exceedance_metrics(summer_events, tau_series, max_gap="20min")

Export as a csv

In [6]:
spring_exceedance.to_csv("spring_events/spring_event_exceedance_metrics.csv", index=False)
summer_exceedance.to_csv("summer_events/summer_event_exceedance_metrics.csv", index=False)

Getting specific JPM times

In [7]:
def add_observed_d50_motion_metrics(events_metrics, d50_motion_times, time_col=None, sample_interval_h=0.25, include_peak_in="rising"):
    df = events_metrics.copy()
    # convert the D50 motion timestamps into a clean DatetimeIndex
    if isinstance(d50_motion_times, pd.DataFrame):
        obs = d50_motion_times.copy()

        if time_col is not None:
            obs_times = pd.to_datetime(obs[time_col], errors="coerce")
        else:
            obs_times = pd.to_datetime(obs.index, errors="coerce")

    else:
        obs_times = pd.to_datetime(d50_motion_times, errors="coerce")

    obs_times = pd.DatetimeIndex(obs_times).dropna().sort_values()

    output_rows = []

    for _, row in df.iterrows():
        result = {}

        start = row["start"]
        end = row["end"]

        # use the peak time used in your exceedance function if available
        if "split_peak_time_used" in row and pd.notna(row["split_peak_time_used"]):
            peak_time = row["split_peak_time_used"]
        else:
            peak_time = row["peak_time"]

        if pd.isna(start) or pd.isna(end) or pd.isna(peak_time) or start >= end:
            result["observed_d50_status"] = "invalid event window"
            output_rows.append(result)
            continue

        # actual D50-motion timestamps inside this event
        event_obs_times = obs_times[
            (obs_times >= start) &
            (obs_times <= end)
        ]

        n_total = len(event_obs_times)

        if n_total == 0:
            result["observed_d50_points_total"] = 0
            result["observed_d50_points_rise"] = 0
            result["observed_d50_points_fall"] = 0

            result["observed_d50_h_total"] = 0.0
            result["observed_d50_h_rise"] = 0.0
            result["observed_d50_h_fall"] = 0.0

            result["observed_d50_delta_time_rise_minus_fall_h"] = 0.0
            result["observed_d50_delta_time_fall_minus_rise_h"] = 0.0
            result["observed_d50_duration_balance"] = np.nan

            result["observed_d50_first_motion_time"] = pd.NaT
            result["observed_d50_last_motion_time"] = pd.NaT
            result["observed_d50_first_motion_relative_to_peak_h"] = np.nan
            result["observed_d50_last_motion_relative_to_peak_h"] = np.nan
            result["observed_d50_centroid_relative_to_peak_h"] = np.nan

            result["observed_d50_status"] = "no observed D50 motion"
            output_rows.append(result)
            continue

        # Split by rising/falling limb
        if include_peak_in == "rising":
            rise_times = event_obs_times[event_obs_times <= peak_time]
            fall_times = event_obs_times[event_obs_times > peak_time]
        elif include_peak_in == "falling":
            rise_times = event_obs_times[event_obs_times < peak_time]
            fall_times = event_obs_times[event_obs_times >= peak_time]
        else:
            raise ValueError("include_peak_in must be 'rising' or 'falling'")

        n_rise = len(rise_times)
        n_fall = len(fall_times)

        T_rise = n_rise * sample_interval_h
        T_fall = n_fall * sample_interval_h
        T_total = n_total * sample_interval_h

        result["observed_d50_points_total"] = n_total
        result["observed_d50_points_rise"] = n_rise
        result["observed_d50_points_fall"] = n_fall

        result["observed_d50_h_total"] = T_total
        result["observed_d50_h_rise"] = T_rise
        result["observed_d50_h_fall"] = T_fall

        # useful sign convention for hysteresis:
        # positive = more actual D50 motion on rising limb
        result["observed_d50_delta_time_rise_minus_fall_h"] = T_rise - T_fall

        # same sign convention as your previous duration_balance:
        # positive = more actual D50 motion on falling limb
        result["observed_d50_delta_time_fall_minus_rise_h"] = T_fall - T_rise

        result["observed_d50_fraction_rise"] = T_rise / T_total
        result["observed_d50_fraction_fall"] = T_fall / T_total

        # -1 = all observed D50 motion on rising limb
        # +1 = all observed D50 motion on falling limb
        result["observed_d50_duration_balance"] = (
            (T_fall - T_rise) / T_total
            if T_total > 0 else np.nan
        )

        # timing metrics relative to peak
        rel_h = (event_obs_times - peak_time).total_seconds() / 3600

        result["observed_d50_first_motion_time"] = event_obs_times.min()
        result["observed_d50_last_motion_time"] = event_obs_times.max()

        result["observed_d50_first_motion_relative_to_peak_h"] = rel_h.min()
        result["observed_d50_last_motion_relative_to_peak_h"] = rel_h.max()
        result["observed_d50_centroid_relative_to_peak_h"] = rel_h.mean()

        result["observed_d50_status"] = "calculated"

        output_rows.append(result)

    observed_metrics = pd.DataFrame(output_rows)

    return pd.concat(
        [df.reset_index(drop=True), observed_metrics.reset_index(drop=True)],
        axis=1
    )

In [9]:
# spring 
d50_motion = pd.read_csv("../data/hydrophones/spring_d50_exceedance_times.csv", parse_dates=["Time"])
spring_exceedance = add_observed_d50_motion_metrics(events_metrics=spring_exceedance, d50_motion_times=d50_motion, time_col="Time", sample_interval_h=0.25)

# summer 
d50_motion_summer = pd.read_csv("../data/hydrophones/summer_d50_exceedance_times.csv", parse_dates=["Time"])
summer_exceedance = add_observed_d50_motion_metrics(events_metrics=summer_exceedance, d50_motion_times=d50_motion_summer, time_col="Time", sample_interval_h=0.25)


FileNotFoundError: [Errno 2] No such file or directory: '../data/hydrophones/spring_d50_exceedance_times.csv'